In [1]:
import pandas as pd
import os.path
import random

In [2]:
#All Function definitions

def generateIDs(keyname, numberOfQuestions):
    '''generates a unique ID for each question using the beginning characters of each question'''
    goodIDs=[]
    with open(keyname) as file:  
        data = file.read()
    for n in range(1,numberOfQuestions+1):
            #print("processing: "+str(n))
            locs=[]
            occursat=find_all(data, str(n)+')')
            for ii in occursat:
                if data[ii-1]=='\r' and data[ii+3]!="_":
                    locs.append(ii)
                    #print(locs)
            z=lambda x: str('0')+str(x) if x<10 else str(x)
            theID=z(n)+"_"+(data[locs[0]+3:locs[0]+25]).strip()
            goodIDs.append(theID)
    return goodIDs

def unduplicate(dup_list):
    '''finds duplicate entries in a list and adds characters to duplicated entries to make them unique.
    works in a roundabout way by converting to a dataframe first , and then converting back to a list in the end'''
    df = pd.DataFrame(dup_list)
    dups=df[df.duplicated(keep='first')]
    if not dups.empty:
        dup_indices=list(dups.index)
        for i in dup_indices:
            df.iat[i,0]=df.iat[i,0]+str(i)
            print("Unduplicating: "+ df.iat[i,0])
    undup_list=list(df.values.flatten())
    return undup_list

def createQuestionIDs(numberOfQuestions):
    '''this will generateIDs unique question IDs for each question from the raw keys'''
    allIDs=[]
    for n in range(0,6):
        filename="key"+str(n)+".txt"
        if (os.path.isfile(filename)): 
            qid=generateIDs(filename,numberOfQuestions)
            allIDs.append(qid)
    allIDs[0]=unduplicate(allIDs[0])
 #The ID's generated are slightly different in each version, so set them all equal to the ID's in version0           
    for n in range(1,numberOfVersions):
        for m in range(0,numberOfQuestions):
            sub=allIDs[0][m][4:20]
            matching = [s for s in allIDs[n] if sub in s]
            found_in=allIDs[n].index(matching[0])
            if allIDs[n][found_in]!=allIDs[0][m]:
                #print(QIDs[n][found_in]+"   changed to    "+QIDs[0][m])
                allIDs[n][found_in]=allIDs[0][m]
    
    return allIDs

def find_all(a_str, sub):
    '''finds all instances of a substring in a string'''
    start = 0
    result=[]
    while True:
        start = a_str.find(sub, start)
        if start == -1: return result
        result.append(start)
        start += len(sub) # use start += 1 to find overlapping matches
    return result

# def cleanKey(key_name):
#     '''retrieves just the "key" part of exported test. 
#     Just reads and returns the key if the questions have already been deleted 
#     writes to a new file with name cleankey'''
#     if(os.path.isfile(key_name)):
#         #print("Scrubbing: "+key_name+" which is :" + str(type(key_name)))
#         with open(key_name) as file:  
#             data = file.read()
#             p=data.rfind('_____')
#             if p!=-1: 
#                 data=data[p+5:]#delete all the junk before "_______"
#                 data=data[data.find('1)'):]#keep all the stuff after the 1)
#             if data.find('\rID:')!=-1:# remove the question ID to a separate column if necessary
#                 data = data.replace('\rID:','\t')
    
#     with open('cleaned_'+key_name, 'w') as file:
#         file.write(data)
        
def getAllKeys():
    '''read all files named key0-key4, make them into strings and and return a dictionary
    containing a list of keys and also the number of questions'''
    keylist=[]

    for n in range(0,6):
        filename="key"+str(n)+".txt"
        print("processing: "+filename)
        if os.path.isfile(filename): 
            cleanKey(filename)#always create a fresh cleankey, to avoid using an outdated one
            keys=pd.read_table("cleaned_"+filename, header=None,usecols=[0])
            keylist.append(makekey(keys))
    numberOfQuestions=keys.shape[0]
    getAllKeysOutput={"keylist":keylist,"numberOfQuestions":numberOfQuestions}
    return getAllKeysOutput


def makekey(key_from_test):
    '''key_from_test is a dataframe formed from reading the answerkey file. This function converts it into a text string with numbers 0-5 representing A-E'''
    keydictionary={"A":"0","B":"1","C":"2","D":"3","E":"4"}
    thekey=""
    for i in range(0,key_from_test.shape[0]):
        #print("At i="+str(i))
        #get the last letter, i.e. the "C" from "1)C" 
        thekey+=keydictionary[key_from_test.iat[i,0][-1:]]
    return thekey


def gradeWithKeylist(keylist, ans, numberOfQuestions, QIDs=[], points=[]):
    '''multiple versions - find the correct key as indicated on the last question on the exam '''
    assert(keylist!=[])
    whichKey=int(ans[-1:])
    key=keylist[whichKey]
    assert len(key)==len(ans)
    missed=""
    invkeydictionary={0:"A",1:"B",2:"C",3:"D",4:"E"}
    rejalt=[1]*numberOfQuestions
    for n in range(0,len(key)-1):
        if key[n]!=ans[n]:
            #print("storing")
            missed+=str(n+1)+", "
            rejalt[n]=0
    if sum(rejalt)==numberOfQuestions:
        missed="ALL CORRECT"
    else:
        missed="v"+invkeydictionary[whichKey]+": "+missed[:len(missed)-2]
    
    if not points:
        points=rejalt
        
    points[-1]=0 #No points for the test version
    score=sum([i*j for i,j in zip(points,rejalt)])
       
    
    if QIDs:
        global analysis_df
        mydict = dict(zip(QIDs[whichKey],rejalt))  
        sortedIDs=sorted(mydict.keys())
        sorted_rejalt=[mydict[k] for k in sortedIDs]
        analysis_df.loc[len(analysis_df)] = sorted_rejalt   
        
    return {'missed':missed, 'score':score}

def process_grades(keylist,data,numberOfQuestions, QIDs):
    '''grades all exams using correct keys, writes questions missed and scores'''
    for NN in range(0, data.shape[0]):
        #print("NN: "+str(NN))
        check1=gradeWithKeylist(keylist, df.iat[NN,2],numberOfQuestions, QIDs)
        data.iat[NN,3]=check1['missed']
        data.iat[NN,4]=check1['score']
    return data

 ####################### SUGGESTED USAGE#################################
# outs = getAllKeys()
# keylist=outs["keylist"]
# numberOfQuestions=outs["numberOfQuestions"] 
# numberOfVersions=len(keylist)

# QIDs=createQuestionIDs(numberOfQuestions)
# analysis_df  = pd.DataFrame(columns = QIDs[0])

In [ ]:
# #Concatenate all the data files into a single one
# filenames = ['1453a.dat', '1453b.dat','1453c.dat']
# with open('1453.dat', 'w') as outfile:
#     for fname in filenames:
#         with open(fname) as infile:
#             outfile.write(infile.read())

In [11]:
#Manually convert the dat file into an excel file using excel. Only extract the serial number,
#name and Answers, and make sure Answers is text.
# Do the following by hand:
#check serial numbers (using remove duplicates)
#manually fill in missing information 
rawdatafilename='Allsmall'
xls_file = pd.ExcelFile(rawdatafilename+'.xlsx', dtype=str)
df = xls_file.parse('Sheet1', header=None, parse_cols=2,names = ["Srl No", "Name", "Answers"])
#parse_cols makes sure that only cols 0,1 and 2 are extracted
df["Missed"] = ""
df["Score"]=0
#numberOfQuestions=len(df.iat[2,2]) #number of questions deduced from the length of the string of answers of (random) student 2

In [12]:
#here is where we run the code
# kl=getAllKeys()
# df=process_grades(df,kl)
outs = getAllKeys()
keylist=outs["keylist"]
numberOfQuestions=outs["numberOfQuestions"] 
numberOfVersions=len(keylist)

QIDs=createQuestionIDs(numberOfQuestions)
analysis_df  = pd.DataFrame(columns = QIDs[0])
df=process_grades(keylist,df,numberOfQuestions, QIDs)

KeyError: '7'

In [ ]:
# #Can find rows where a null value exists
# null_columns=df.columns[df.isnull().any()]
# print(df[df.isnull().any(axis=1)][null_columns].head())

In [13]:
#Grades and displays the N'th entry in the list using a keylist
N=random.randint(1,df.shape[0])
check1=gradeWithKeylist(keylist, df.iat[N,2],numberOfQuestions, QIDs)
print("Row "+ str(N)+": "+df.iat[N,1]+", "+str(df.iat[N,0])+" missed "+str(check1['missed'])+ " , and scored "+str(check1['score']))

NameError: name 'keylist' is not defined

In [ ]:
# writing to csv
df.to_csv(rawdatafilename+'_processed.csv')
analysis_df.to_csv(rawdatafilename+'_analysis.csv')
#### Don't opoen this file directly in excel, since the "Answers" column gets messed up. Import it using the import wizard. 

In [3]:
outs = getAllKeys()

processing: key0.txt


KeyError: '7'

In [3]:
key_name='key0.txt'
if(os.path.isfile(key_name)):
    #print("Scrubbing: "+key_name+" which is :" + str(type(key_name)))
    with open(key_name) as file:  
        data = file.read()

In [4]:
p=data.rfind('_____')

In [6]:
data1=data[p+5:]#delete all the junk before "_______"

In [7]:
data1

'\nA)      B)      C)      D)\n\n\n\n1) B\nID: up13 14.1-7\n\n2) A\nID: up13 9.1-5+\n\n3) D\nID: up13 8.2-29\n\n4) A\nID: up13 8.2-5\n\n5) D\nID: up13 13.2-16+\n\n6) A\nID: up13 16.2-41\n\n7) C\nID: up13 6.2-36+\n\n8) C\nID: up13 15.2-12+\n\n9) E\nID: up13 1.2-80\n\n10) D\nID: up13 11.2-13+\n\n11) A\nID: up13 6.1-8\n\n12) E\nID: USER-1\n\n13) C\nID: up13 14.2-26\n\n14) B\nID: up13 15.2-5\n\n15) D\nID: up13 11.1-3\n\n16) B\nID: up13 14.1-2\n\n17) E\nID: USER-3\n\n18) C\nID: up13 16.2-31\n\n19) B\nID: USER-3\n\n20) D\nID: up13 10.2-19\n\n21) C\nID: up13 15.2-16\n\n22) C\nID: up13 16.2-10\n\n23) D\nID: USER-4\n\n24) B\nID: up13 15.1-8\n\n25) C\nID: up13 5.2-52\n\n26) A\nID: USER-1\n'

In [8]:
data1=data1[data1.find('1)'):]#keep all the stuff after the 1)
data1

'1) B\nID: up13 14.1-7\n\n2) A\nID: up13 9.1-5+\n\n3) D\nID: up13 8.2-29\n\n4) A\nID: up13 8.2-5\n\n5) D\nID: up13 13.2-16+\n\n6) A\nID: up13 16.2-41\n\n7) C\nID: up13 6.2-36+\n\n8) C\nID: up13 15.2-12+\n\n9) E\nID: up13 1.2-80\n\n10) D\nID: up13 11.2-13+\n\n11) A\nID: up13 6.1-8\n\n12) E\nID: USER-1\n\n13) C\nID: up13 14.2-26\n\n14) B\nID: up13 15.2-5\n\n15) D\nID: up13 11.1-3\n\n16) B\nID: up13 14.1-2\n\n17) E\nID: USER-3\n\n18) C\nID: up13 16.2-31\n\n19) B\nID: USER-3\n\n20) D\nID: up13 10.2-19\n\n21) C\nID: up13 15.2-16\n\n22) C\nID: up13 16.2-10\n\n23) D\nID: USER-4\n\n24) B\nID: up13 15.1-8\n\n25) C\nID: up13 5.2-52\n\n26) A\nID: USER-1\n'

In [9]:
endline=['\rID:', '\nID:']
for pp in endline:
    if data1.find(pp)!=-1:# remove the question ID to a separate column if necessary
        data1 = data1.replace(pp,'\t')

In [10]:
data1

'1) B\t up13 14.1-7\n\n2) A\t up13 9.1-5+\n\n3) D\t up13 8.2-29\n\n4) A\t up13 8.2-5\n\n5) D\t up13 13.2-16+\n\n6) A\t up13 16.2-41\n\n7) C\t up13 6.2-36+\n\n8) C\t up13 15.2-12+\n\n9) E\t up13 1.2-80\n\n10) D\t up13 11.2-13+\n\n11) A\t up13 6.1-8\n\n12) E\t USER-1\n\n13) C\t up13 14.2-26\n\n14) B\t up13 15.2-5\n\n15) D\t up13 11.1-3\n\n16) B\t up13 14.1-2\n\n17) E\t USER-3\n\n18) C\t up13 16.2-31\n\n19) B\t USER-3\n\n20) D\t up13 10.2-19\n\n21) C\t up13 15.2-16\n\n22) C\t up13 16.2-10\n\n23) D\t USER-4\n\n24) B\t up13 15.1-8\n\n25) C\t up13 5.2-52\n\n26) A\t USER-1\n'

In [12]:
def cleanKey(key_name):
    '''retrieves just the "key" part of exported test. 
    Just reads and returns the key if the questions have already been deleted 
    writes to a new file with name cleankey'''
    if(os.path.isfile(key_name)):
        #print("Scrubbing: "+key_name+" which is :" + str(type(key_name)))
        with open(key_name) as file:  
            data = file.read()
            p=data.rfind('_____')
            if p!=-1: 
                data=data[p+5:]#delete all the junk before "_______"
                data=data[data.find('1)'):]#keep all the stuff after the 1)
    endline=['\rID:', '\nID:']
    for pp in endline:
        if data.find(pp)!=-1:# remove the question ID to a separate column if necessary
            data = data.replace(pp,'\t')
    with open('cleaned_'+key_name, 'w') as file:
        file.write(data)

In [14]:
key='key0.txt'

In [ ]:
clean